<a href="https://colab.research.google.com/github/yuli894/DL_learning/blob/main/Transformer_test_protein.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math                         # 导入数学运算相关库
import random                       # 导入随机数生成库
import numpy as np                  # 导入NumPy库，用于数组操作
import torch                        # 导入PyTorch主库
import torch.nn as nn               # 导入神经网络模块
import torch.optim as optim         # 导入优化器模块
from torch.utils.data import Dataset, DataLoader  # 导入数据集和数据加载工具
import torch.nn.functional as F     # 导入torch.nn.functional，包含各种激活和损失函数

# 设置随机种子，确保实验可重复
SEED = 42
random.seed(SEED)                   # Python随机数种子设置
np.random.seed(SEED)                # NumPy随机数种子设置
torch.manual_seed(SEED)             # CPU随机数种子设置
torch.cuda.manual_seed_all(SEED)    # 如果使用CUDA，设置所有GPU的随机数种子

# -------------------------------
# 定义蛋白质相关常量
# -------------------------------
# 标准20种氨基酸，注意 0 保留给padding，因此索引从1开始
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
VOCAB_SIZE = len(AMINO_ACIDS) + 1    # 词汇表大小：20种氨基酸加一个padding字符

# Kyte-Doolittle疏水性评分字典，用于衡量每个氨基酸的疏水性
HYDROPHOBICITY = {
    'A': 1.8,  'C': 2.5,  'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5,  'K': -3.9, 'L': 3.8,
    'M': 1.9,  'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2,  'W': -0.9, 'Y': -1.3
}
# 将氨基酸字符映射为索引，注意索引0保留给padding
char_to_index = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}

# -------------------------------
# 超参数设置
# -------------------------------
EMBED_DIM = 128                  # 嵌入向量维度
NUM_HEADS = 4                    # Transformer多头注意力头数
HIDDEN_DIM = 64                  # 前馈网络隐藏层维度
NUM_ENCODER_LAYERS = 2           # Transformer编码器层数
MAX_SEQ_LEN = 100                # 最大蛋白质序列长度（可根据实际情况调节）
BATCH_SIZE = 32                  # 批处理大小
NUM_EPOCHS = 20                  # 训练轮数
LEARNING_RATE = 1e-3             # 学习率

# =======================
# 1. 合成蛋白质数据集定义
# =======================
class SyntheticProteinDataset(Dataset):
    def __init__(self, num_samples):
        self.data = []           # 存储蛋白质序列数据
        self.labels = []         # 存储对应标签
        # 设定一个阈值，用于二分类（例如：平均疏水性 > 0 则归为1，否则为0）
        threshold = 0.0
        for _ in range(num_samples):
            # 模拟蛋白质序列长度，范围为30到MAX_SEQ_LEN
            seq_len = random.randint(30, MAX_SEQ_LEN)
            # 随机选择氨基酸构造蛋白质序列，转换为对应的索引表示
            sequence = [char_to_index[random.choice(AMINO_ACIDS)] for _ in range(seq_len)]
            # 根据Kyte-Doolittle疏水性评分计算整个序列的平均疏水性
            hydro_score = np.mean([HYDROPHOBICITY[AMINO_ACIDS[idx-1]] for idx in sequence])
            # 根据平均疏水性判断类别，疏水性大于阈值归为1，否则归为0
            label = 1 if hydro_score > threshold else 0
            self.data.append(sequence)   # 将生成的序列添加到数据列表中
            self.labels.append(label)      # 将对应的标签添加到标签列表中

    def __len__(self):
        return len(self.data)    # 返回数据集中样本的数量

    def __getitem__(self, idx):
        # 根据索引返回对应的蛋白质序列和标签，并转换为PyTorch tensor格式
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# 定义collate函数，用于批次内序列的padding对齐，同时生成mask
def collate_fn(batch):
    data, labels = zip(*batch)                     # 解压批次数据和标签
    lengths = [len(seq) for seq in data]             # 获取每个序列的长度
    max_len = max(lengths)                           # 找到批次内最长的序列长度
    # 对每个序列进行padding操作，使其长度统一为max_len，padding值为0
    padded = [F.pad(seq, (0, max_len - len(seq)), value=0) for seq in data]
    padded_seqs = torch.stack(padded)              # 将padding后的序列堆叠成一个张量
    mask = (padded_seqs == 0)                        # mask标记填充部分，方便后续忽略
    return padded_seqs, mask, torch.stack(labels)    # 返回序列张量、mask和标签张量

# 生成训练数据集和测试数据集，分别包含10000个和2000个样本
train_dataset = SyntheticProteinDataset(1000)
test_dataset = SyntheticProteinDataset(200)
# 根据数据集创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# =======================
# 2. 位置编码模块（保持不变）
# =======================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)                   # 初始化位置编码矩阵，形状为[max_len, d_model]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # 生成位置索引，[max_len, 1]
        # 计算用于位置编码的缩放因子，应用指数衰减
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # 对偶数索引应用sin函数
        pe[:, 0::2] = torch.sin(position * div_term)
        # 对奇数索引应用cos函数
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                                 # 增加batch维度，形状变为[1, max_len, d_model]
        self.register_buffer('pe', pe)                       # 将pe注册为buffer，模型保存时会一起保存但不更新参数

    def forward(self, x):
        # x形状为[batch_size, seq_len, d_model]
        # 将输入x与对应位置编码相加，并返回相加后的结果
        return x + self.pe[:, :x.size(1), :]

# =======================
# 3. Transformer分类器
# =======================
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes, max_seq_len):
        super().__init__()
        # 嵌入层：将每个token（氨基酸索引）映射到一个嵌入向量
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # 位置编码模块，添加序列位置信息
        self.pos_encoder = PositionalEncoding(embed_dim, max_seq_len)
        # 定义一个Transformer编码器层
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim)
        # 堆叠多个Transformer编码器层
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # 全连接层，将Transformer的输出映射到类别数（这里二分类，所以num_classes=2）
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, src, src_key_padding_mask):
        x = self.embedding(src)                # 将输入token索引转换为嵌入向量，形状变为[batch, seq, embed_dim]
        x = self.pos_encoder(x)                # 添加位置编码
        x = x.transpose(0, 1)                  # 转置为Transformer要求的格式，[seq, batch, embed_dim]
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)  # 通过Transformer编码器处理
        x = x.mean(dim=0)                      # 对序列维度做平均池化，获得整个序列的全局表示
        return self.fc(x)                      # 通过全连接层输出最终分类结果

# -------------------------------
# 初始化模型与设备
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # 若有GPU则使用GPU，否则使用CPU
model = TransformerClassifier(
    vocab_size=VOCAB_SIZE,             # 词汇表大小
    embed_dim=EMBED_DIM,               # 嵌入向量维度
    num_heads=NUM_HEADS,               # 多头注意力头数
    hidden_dim=HIDDEN_DIM,             # 前馈隐藏层维度
    num_layers=NUM_ENCODER_LAYERS,     # Transformer编码器层数
    num_classes=2,                     # 分类任务类别数（二分类）
    max_seq_len=MAX_SEQ_LEN            # 最大序列长度
).to(device)                          # 将模型移动到指定设备（GPU或CPU）
print("是否使用GPU:", next(model.parameters()).is_cuda)  # 打印模型是否使用GPU

# =======================
# 4. 训练与评估函数
# =======================
criterion = nn.CrossEntropyLoss()      # 定义交叉熵损失函数，适用于多分类任务
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)  # 定义Adam优化器

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()                      # 切换模型到训练模式
    total_loss, correct, total = 0, 0, 0  # 初始化累计损失、正确预测数和样本总数
    for src, mask, labels in loader:
        src, mask, labels = src.to(device), mask.to(device), labels.to(device)  # 将数据移动到指定设备
        optimizer.zero_grad()          # 梯度清零
        outputs = model(src, src_key_padding_mask=mask)  # 前向传播，获得模型输出
        loss = criterion(outputs, labels)   # 计算损失值
        loss.backward()               # 反向传播，计算梯度
        optimizer.step()              # 更新模型参数

        total_loss += loss.item() * src.size(0)   # 累计损失，乘以样本数
        correct += (outputs.argmax(1) == labels).sum().item()  # 统计预测正确的样本数
        total += src.size(0)           # 统计总样本数
    return total_loss / total, correct / total  # 返回平均损失和准确率

def evaluate(model, loader, criterion, device):
    model.eval()                       # 切换模型到评估模式
    total_loss, correct, total = 0, 0, 0  # 初始化累计损失、正确预测数和样本总数
    with torch.no_grad():              # 禁用梯度计算，加快评估速度
        for src, mask, labels in loader:
            src, mask, labels = src.to(device), mask.to(device), labels.to(device)  # 数据移动到指定设备
            outputs = model(src, src_key_padding_mask=mask)  # 前向传播
            loss = criterion(outputs, labels)   # 计算损失
            total_loss += loss.item() * src.size(0)  # 累计损失
            correct += (outputs.argmax(1) == labels).sum().item()  # 统计正确预测数
            total += src.size(0)       # 累计样本数
    return total_loss / total, correct / total  # 返回平均损失和准确率

# =======================
# 5. 训练主循环
# =======================
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)  # 训练一个epoch
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)      # 在测试集上评估
    # 打印当前轮的训练和测试损失及准确率
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
          f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}")

# =======================
# 6. 模型预测示例
# =======================
model.eval()                          # 切换模型到评估模式
sample_src, sample_mask, sample_label = next(iter(test_loader))  # 从测试集获取一个批次样本
sample_src, sample_mask = sample_src.to(device), sample_mask.to(device)  # 数据移动到设备上
with torch.no_grad():                 # 禁用梯度计算
    output = model(sample_src, src_key_padding_mask=sample_mask)  # 模型前向传播预测输出
    pred = output.argmax(1)           # 获取预测结果（取概率最大的类别索引）
# 打印真实标签与模型预测标签
print("样本真实标签：", sample_label.numpy())
print("模型预测标签：", pred.cpu().numpy())


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


是否使用GPU: False
Epoch 1: Train Loss=0.1717, Train Acc=0.9362, Test Loss=0.1252, Test Acc=0.9570
Epoch 2: Train Loss=0.0785, Train Acc=0.9683, Test Loss=0.0423, Test Acc=0.9840
Epoch 3: Train Loss=0.0639, Train Acc=0.9738, Test Loss=0.0506, Test Acc=0.9815
Epoch 4: Train Loss=0.0581, Train Acc=0.9749, Test Loss=0.0533, Test Acc=0.9740
Epoch 5: Train Loss=0.0459, Train Acc=0.9820, Test Loss=0.0697, Test Acc=0.9740
Epoch 6: Train Loss=0.0524, Train Acc=0.9787, Test Loss=0.0431, Test Acc=0.9785
Epoch 7: Train Loss=0.0423, Train Acc=0.9821, Test Loss=0.0425, Test Acc=0.9820
Epoch 8: Train Loss=0.0461, Train Acc=0.9801, Test Loss=0.0316, Test Acc=0.9875
Epoch 9: Train Loss=0.0424, Train Acc=0.9820, Test Loss=0.0691, Test Acc=0.9755
Epoch 10: Train Loss=0.0446, Train Acc=0.9812, Test Loss=0.0298, Test Acc=0.9875
Epoch 11: Train Loss=0.0424, Train Acc=0.9819, Test Loss=0.0551, Test Acc=0.9740
Epoch 12: Train Loss=0.0338, Train Acc=0.9862, Test Loss=0.0403, Test Acc=0.9840
Epoch 13: Train Loss=0